<a href="https://colab.research.google.com/github/ibrahimbarghout/robust-ecg-domain-generalization/blob/main/notebooks/11_ECG_Noise_Robustness_And_Signal_Quality_Generalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# NOTEBOOK 11 — NOISE ROBUSTNESS & SIGNAL-QUALITY GENERALIZATION
# STEP 11A — ENVIRONMENT & FROZEN BASELINE VERIFICATION
# ============================================================

!pip install wfdb -q
from google.colab import drive
drive.mount("/content/drive")
import os
import json
import hashlib
import numpy as np
import pandas as pd
import torch
import wfdb

print("=" * 70)
print("STEP 11A — ENVIRONMENT & FROZEN BASELINE VERIFICATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_PATH_11A = (
    "/content/drive/MyDrive/PTB-XL Research Project"
)

DATA_PATH_11A = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
)

RESULTS_PATH_11A = (
    "/content/drive/MyDrive/PTB-XL Research Project/results"
)

DEEP_LEARNING_PATH_11A = os.path.join(
    RESULTS_PATH_11A,
    "deep_learning"
)

NOISE_RESULTS_PATH_11A = os.path.join(
    RESULTS_PATH_11A,
    "noise_robustness"
)

FROZEN_BASELINE_PATH_11A = os.path.join(
    DEEP_LEARNING_PATH_11A,
    "ecg_resnet1d_frozen_baseline.json"
)

CHECKPOINT_PATH_11A = os.path.join(
    DEEP_LEARNING_PATH_11A,
    "ecg_resnet1d_best_validation.pt"
)

TEST_RESULTS_PATH_11A = os.path.join(
    DEEP_LEARNING_PATH_11A,
    "ecg_resnet1d_test_results.csv"
)

# ------------------------------------------------------------
# 2. Verify paths
# ------------------------------------------------------------

required_paths_11A = {
    "project": PROJECT_PATH_11A,
    "dataset": DATA_PATH_11A,
    "results": RESULTS_PATH_11A,
    "deep_learning_results": DEEP_LEARNING_PATH_11A,
    "frozen_baseline": FROZEN_BASELINE_PATH_11A,
    "checkpoint": CHECKPOINT_PATH_11A,
    "test_results": TEST_RESULTS_PATH_11A
}

print("\nPATH VERIFICATION")
print("-" * 70)

for name_11A, path_11A in required_paths_11A.items():

    exists_11A = os.path.exists(path_11A)

    print(
        f"{name_11A:<25}: "
        f"{'PASS' if exists_11A else 'FAIL'}"
    )

    if not exists_11A:
        raise FileNotFoundError(
            f"Required path not found: {path_11A}"
        )

# ------------------------------------------------------------
# 3. Create Notebook 11 results directory
# ------------------------------------------------------------

os.makedirs(
    NOISE_RESULTS_PATH_11A,
    exist_ok=True
)

print(
    "\nNoise robustness results directory: "
    f"{NOISE_RESULTS_PATH_11A}"
)

# ------------------------------------------------------------
# 4. Load frozen baseline record
# ------------------------------------------------------------

with open(
    FROZEN_BASELINE_PATH_11A,
    "r"
) as file_11A:

    frozen_baseline_11A = json.load(
        file_11A
    )

print("\nFROZEN BASELINE")
print("-" * 70)

print(
    f"Status:                   "
    f"{frozen_baseline_11A['status']}"
)

print(
    f"Model:                    "
    f"{frozen_baseline_11A['model']}"
)

print(
    f"Architecture:             "
    f"{frozen_baseline_11A['architecture']}"
)

print(
    f"Representation:           "
    f"{frozen_baseline_11A['input_representation']}"
)

print(
    f"Parameters:               "
    f"{frozen_baseline_11A['parameter_count']:,}"
)

print(
    f"Selected checkpoint:      "
    f"Epoch "
    f"{frozen_baseline_11A['selected_checkpoint_epoch']}"
)

print(
    f"Validation macro-AUROC:   "
    f"{frozen_baseline_11A['validation_macro_AUROC']:.6f}"
)

print(
    f"Locked test macro-AUROC:  "
    f"{frozen_baseline_11A['test_macro_AUROC']:.6f}"
)

# ------------------------------------------------------------
# 5. Verify frozen baseline properties
# ------------------------------------------------------------

assert frozen_baseline_11A["status"] == "FROZEN"

assert frozen_baseline_11A[
    "model"
] == "ECGResNet1D"

assert frozen_baseline_11A[
    "architecture"
] == "1D ResNet"

assert frozen_baseline_11A[
    "input_representation"
] == "raw 12-lead ECG waveform"

assert frozen_baseline_11A[
    "input_shape"
] == [12, 5000]

assert frozen_baseline_11A[
    "sampling_frequency_hz"
] == 500

assert frozen_baseline_11A[
    "duration_seconds"
] == 10

assert frozen_baseline_11A[
    "number_of_targets"
] == 5

assert frozen_baseline_11A[
    "targets"
] == [
        "MI",
        "STTC",
        "CD",
        "HYP",
        "NORM"
    ]

assert frozen_baseline_11A[
    "parameter_count"
] == 8739973

assert frozen_baseline_11A[
    "selected_checkpoint_epoch"
] == 17

assert frozen_baseline_11A[
    "test_set_used_for_model_selection"
] is False

assert frozen_baseline_11A[
    "test_set_used_for_threshold_tuning"
] is False

assert frozen_baseline_11A[
    "test_set_used_for_hyperparameter_tuning"
] is False

# ------------------------------------------------------------
# 6. Verify checkpoint exists and is readable
# ------------------------------------------------------------

checkpoint_11A = torch.load(
    CHECKPOINT_PATH_11A,
    map_location="cpu"
)

assert isinstance(
    checkpoint_11A,
    dict
)

assert checkpoint_11A[
    "epoch"
] == 17

assert checkpoint_11A[
    "target_columns"
] == [
    "MI",
    "STTC",
    "CD",
    "HYP",
    "NORM"
]

assert checkpoint_11A[
    "model_parameters"
] == 8739973

print("\nCHECKPOINT VERIFICATION")
print("-" * 70)

print(
    f"Checkpoint epoch:        "
    f"{checkpoint_11A['epoch']}"
)

print(
    f"Checkpoint parameters:   "
    f"{checkpoint_11A['model_parameters']:,}"
)

print(
    f"Checkpoint val AUROC:    "
    f"{checkpoint_11A['best_validation_macro_auroc']:.6f}"
)

# ------------------------------------------------------------
# 7. Verify locked test results
# ------------------------------------------------------------

test_results_11A = pd.read_csv(
    TEST_RESULTS_PATH_11A
)

assert len(test_results_11A) == 5

assert set(
    test_results_11A["target"]
) == {
    "MI",
    "STTC",
    "CD",
    "HYP",
    "NORM"
}

test_macro_auroc_11A = float(
    test_results_11A["AUROC"].mean()
)

assert abs(
    test_macro_auroc_11A - 0.930060
) < 1e-5

print("\nLOCKED TEST RESULT VERIFICATION")
print("-" * 70)

print(
    f"Test ECGs:               "
    f"2,198"
)

print(
    f"Test macro-AUROC:        "
    f"{test_macro_auroc_11A:.6f}"
)

# ------------------------------------------------------------
# 8. Environment
# ------------------------------------------------------------

print("\nENVIRONMENT")
print("-" * 70)

print(
    f"Python:                   "
    f"{__import__('sys').version.split()[0]}"
)

print(
    f"NumPy:                    "
    f"{np.__version__}"
)

print(
    f"Pandas:                   "
    f"{pd.__version__}"
)

print(
    f"PyTorch:                  "
    f"{torch.__version__}"
)

print(
    f"WFDB:                     "
    f"{wfdb.__version__}"
)

print(
    f"CUDA available:           "
    f"{torch.cuda.is_available()}"
)

if torch.cuda.is_available():

    print(
        f"GPU:                      "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        f"CUDA version:             "
        f"{torch.version.cuda}"
    )

# ------------------------------------------------------------
# 9. Record checkpoint hash
# ------------------------------------------------------------

sha256_11A = hashlib.sha256()

with open(
    CHECKPOINT_PATH_11A,
    "rb"
) as file_11A:

    for chunk_11A in iter(
        lambda: file_11A.read(1024 * 1024),
        b""
    ):
        sha256_11A.update(
            chunk_11A
        )

checkpoint_hash_11A = sha256_11A.hexdigest()

print("\nFROZEN CHECKPOINT SHA-256")
print("-" * 70)
print(checkpoint_hash_11A)

# ------------------------------------------------------------
# 10. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 11A STATUS: PASS")
print("Frozen Notebook 10 baseline verified.")
print("No model training or test-set modification performed.")
print("=" * 70)

Mounted at /content/drive
STEP 11A — ENVIRONMENT & FROZEN BASELINE VERIFICATION

PATH VERIFICATION
----------------------------------------------------------------------
project                  : PASS
dataset                  : PASS
results                  : PASS
deep_learning_results    : PASS
frozen_baseline          : PASS
checkpoint               : PASS
test_results             : PASS

Noise robustness results directory: /content/drive/MyDrive/PTB-XL Research Project/results/noise_robustness

FROZEN BASELINE
----------------------------------------------------------------------
Status:                   FROZEN
Model:                    ECGResNet1D
Architecture:             1D ResNet
Representation:           raw 12-lead ECG waveform
Parameters:               8,739,973
Selected checkpoint:      Epoch 17
Validation macro-AUROC:   0.934005
Locked test macro-AUROC:  0.930060

CHECKPOINT VERIFICATION
----------------------------------------------------------------------
Checkpoint epo

In [5]:
import os
import json
import numpy as np
import pandas as pd

# ============================================================
# STEP 11B — NOISE ROBUSTNESS EVALUATION PROTOCOL
# ============================================================

NOISE_RESULTS_PATH_11B = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "results/noise_robustness"
)

os.makedirs(NOISE_RESULTS_PATH_11B, exist_ok=True)

TARGET_COLUMNS_11B = ["MI", "STTC", "CD", "HYP", "NORM"]

TEST_N_11B = 2198
SAMPLING_FREQUENCY_11B = 500
SIGNAL_DURATION_SECONDS_11B = 10
SIGNAL_LENGTH_11B = 5000
N_LEADS_11B = 12

NOISE_LEVELS_11B = [
    {
        "condition": "clean",
        "noise_type": "none",
        "snr_db": None,
        "description": "Original unmodified ECG waveform"
    },
    {
        "condition": "snr_30db",
        "noise_type": "gaussian",
        "snr_db": 30,
        "description": "Additive Gaussian noise at 30 dB SNR"
    },
    {
        "condition": "snr_20db",
        "noise_type": "gaussian",
        "snr_db": 20,
        "description": "Additive Gaussian noise at 20 dB SNR"
    },
    {
        "condition": "snr_10db",
        "noise_type": "gaussian",
        "snr_db": 10,
        "description": "Additive Gaussian noise at 10 dB SNR"
    },
    {
        "condition": "snr_0db",
        "noise_type": "gaussian",
        "snr_db": 0,
        "description": "Additive Gaussian noise at 0 dB SNR"
    }
]

PROTOCOL_11B = {
    "purpose": (
        "Evaluate robustness of the frozen raw-waveform ResNet-1D "
        "under controlled additive Gaussian noise."
    ),
    "model": "ECGResNet1D",
    "model_status": "frozen",
    "checkpoint_epoch": 17,
    "representation": "raw 12-lead ECG waveform",
    "input_shape": [N_LEADS_11B, SIGNAL_LENGTH_11B],
    "sampling_frequency_hz": SAMPLING_FREQUENCY_11B,
    "duration_seconds": SIGNAL_DURATION_SECONDS_11B,
    "targets": TARGET_COLUMNS_11B,
    "test_ecg_count": TEST_N_11B,
    "split": "patient-independent frozen Notebook 10 test split",
    "noise_domain": "synthetic controlled perturbation",
    "noise_type": "additive zero-mean Gaussian noise",
    "snr_definition": (
        "10*log10(signal_power/noise_power), "
        "with noise power matched to the clean signal power"
    ),
    "noise_levels_db": [30, 20, 10, 0],
    "clean_condition_included": True,
    "model_retraining": False,
    "threshold_tuning": False,
    "test_set_used_for_model_selection": False,
    "random_seed": 42,
    "primary_metric": "macro_AUROC",
    "secondary_metrics": [
        "macro_AUPRC",
        "per_class_AUROC",
        "per_class_AUPRC",
        "per_class_F1",
        "per_class_sensitivity",
        "per_class_specificity",
        "degradation_from_clean"
    ],
    "interpretation": (
        "This experiment measures predictive robustness, not clinical "
        "performance or clinical noise tolerance."
    )
}

PROTOCOL_PATH_11B = os.path.join(
    NOISE_RESULTS_PATH_11B,
    "step11B_noise_robustness_protocol.json"
)

with open(PROTOCOL_PATH_11B, "w", encoding="utf-8") as f:
    json.dump(PROTOCOL_11B, f, indent=2)

print("=" * 70)
print("STEP 11B — NOISE ROBUSTNESS EVALUATION PROTOCOL")
print("=" * 70)
print()
print("Model:", PROTOCOL_11B["model"])
print("Model status:", PROTOCOL_11B["model_status"])
print("Checkpoint epoch:", PROTOCOL_11B["checkpoint_epoch"])
print("Test ECGs:", PROTOCOL_11B["test_ecg_count"])
print("Input shape:", PROTOCOL_11B["input_shape"])
print("Sampling frequency:", f'{PROTOCOL_11B["sampling_frequency_hz"]} Hz')
print()
print("Noise conditions:")

for condition in NOISE_LEVELS_11B:
    print(
        f'  {condition["condition"]:>10} | '
        f'{condition["description"]}'
    )

print()
print("Retraining:", PROTOCOL_11B["model_retraining"])
print("Threshold tuning:", PROTOCOL_11B["threshold_tuning"])
print(
    "Test used for model selection:",
    PROTOCOL_11B["test_set_used_for_model_selection"]
)
print()
print("Protocol saved to:")
print(PROTOCOL_PATH_11B)
print()
print("STEP 11B STATUS: PASS")

STEP 11B — NOISE ROBUSTNESS EVALUATION PROTOCOL

Model: ECGResNet1D
Model status: frozen
Checkpoint epoch: 17
Test ECGs: 2198
Input shape: [12, 5000]
Sampling frequency: 500 Hz

Noise conditions:
       clean | Original unmodified ECG waveform
    snr_30db | Additive Gaussian noise at 30 dB SNR
    snr_20db | Additive Gaussian noise at 20 dB SNR
    snr_10db | Additive Gaussian noise at 10 dB SNR
     snr_0db | Additive Gaussian noise at 0 dB SNR

Retraining: False
Threshold tuning: False
Test used for model selection: False

Protocol saved to:
/content/drive/MyDrive/PTB-XL Research Project/results/noise_robustness/step11B_noise_robustness_protocol.json

STEP 11B STATUS: PASS


In [6]:
import os
import numpy as np
import pandas as pd
import wfdb
from scipy import signal

# ============================================================
# STEP 11C — PTB-XL NATURAL SIGNAL-QUALITY CHARACTERIZATION
# ============================================================

DATA_PATH_11C = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
)

RESULTS_PATH_11C = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "results/noise_robustness"
)

SPLIT_PATH_11C = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "results/ptbxl_patient_independent_split.csv"
)

METADATA_PATH_11C = os.path.join(
    DATA_PATH_11C,
    "ptbxl_database.csv"
)

QUALITY_OUTPUT_PATH_11C = os.path.join(
    RESULTS_PATH_11C,
    "step11C_test_signal_quality.csv"
)

# ------------------------------------------------------------
# Load frozen patient-independent split
# ------------------------------------------------------------

split_df_11C = pd.read_csv(SPLIT_PATH_11C)

test_ids_11C = (
    split_df_11C.loc[
        split_df_11C["split"] == "test",
        "ecg_id"
    ]
    .astype(int)
    .tolist()
)

assert len(test_ids_11C) == 2198
assert len(set(test_ids_11C)) == 2198

# ------------------------------------------------------------
# Load PTB-XL metadata
# ------------------------------------------------------------

meta_11C = pd.read_csv(METADATA_PATH_11C)

meta_11C["ecg_id"] = meta_11C["ecg_id"].astype(int)

test_meta_11C = (
    meta_11C[
        meta_11C["ecg_id"].isin(test_ids_11C)
    ]
    .copy()
)

assert len(test_meta_11C) == 2198

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def rms_11C(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.sqrt(np.mean(x ** 2)))


def baseline_wander_ratio_11C(x, fs=500):
    """
    Estimate low-frequency baseline content using a
    0.5 Hz low-pass Butterworth filter.

    Ratio = RMS(low-frequency component) / RMS(signal).
    This is a descriptive signal-quality measure, not
    a clinical quality label.
    """
    x = np.asarray(x, dtype=np.float64)

    total_rms = rms_11C(x)

    if total_rms == 0:
        return 0.0

    b, a = signal.butter(
        4,
        0.5,
        btype="lowpass",
        fs=fs
    )

    low_freq = signal.filtfilt(
        b,
        a,
        x
    )

    return float(rms_11C(low_freq) / total_rms)


def high_frequency_ratio_11C(x, fs=500, cutoff=40):
    """
    Estimate high-frequency content above 40 Hz.

    Ratio = RMS(high-frequency component) / RMS(signal).
    """
    x = np.asarray(x, dtype=np.float64)

    total_rms = rms_11C(x)

    if total_rms == 0:
        return 0.0

    b, a = signal.butter(
        4,
        cutoff,
        btype="highpass",
        fs=fs
    )

    high_freq = signal.filtfilt(
        b,
        a,
        x
    )

    return float(rms_11C(high_freq) / total_rms)


def signal_range_11C(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.ptp(x))


# ------------------------------------------------------------
# Characterize every test ECG
# ------------------------------------------------------------

rows_11C = []

print("=" * 70)
print("STEP 11C — NATURAL PTB-XL SIGNAL-QUALITY CHARACTERIZATION")
print("=" * 70)
print()
print(f"Test ECGs: {len(test_ids_11C)}")
print("Leads per ECG: 12")
print("Sampling frequency: 500 Hz")
print()

for i, (_, row) in enumerate(test_meta_11C.iterrows(), start=1):

    ecg_id = int(row["ecg_id"])
    filename = row["filename_hr"]

    record_path = os.path.join(
        DATA_PATH_11C,
        filename
    )

    record = wfdb.rdrecord(record_path)

    x = np.asarray(
        record.p_signal,
        dtype=np.float64
    )

    assert x.shape == (5000, 12), (
        f"Unexpected shape for ECG {ecg_id}: {x.shape}"
    )

    assert np.isfinite(x).all(), (
        f"Non-finite values found in ECG {ecg_id}"
    )

    lead_rms = []
    lead_ranges = []
    lead_bw = []
    lead_hf = []

    for lead_idx in range(12):

        lead = x[:, lead_idx]

        lead_rms.append(rms_11C(lead))
        lead_ranges.append(signal_range_11C(lead))
        lead_bw.append(
            baseline_wander_ratio_11C(lead)
        )
        lead_hf.append(
            high_frequency_ratio_11C(lead)
        )

    rows_11C.append({
        "ecg_id": ecg_id,
        "patient_id": row["patient_id"],
        "mean_lead_rms_mV": float(np.mean(lead_rms)),
        "median_lead_rms_mV": float(np.median(lead_rms)),
        "mean_lead_range_mV": float(np.mean(lead_ranges)),
        "median_lead_range_mV": float(np.median(lead_ranges)),
        "mean_baseline_wander_ratio": float(np.mean(lead_bw)),
        "max_baseline_wander_ratio": float(np.max(lead_bw)),
        "mean_high_frequency_ratio": float(np.mean(lead_hf)),
        "max_high_frequency_ratio": float(np.max(lead_hf)),
        "zero_rms_leads": int(
            np.sum(np.asarray(lead_rms) == 0)
        )
    })

    if i % 100 == 0 or i == len(test_meta_11C):
        print(
            f"Processed {i:4d}/{len(test_meta_11C)} ECGs"
        )

# ------------------------------------------------------------
# Save result
# ------------------------------------------------------------

quality_df_11C = pd.DataFrame(rows_11C)

assert len(quality_df_11C) == 2198
assert quality_df_11C["ecg_id"].nunique() == 2198
assert np.isfinite(
    quality_df_11C.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

quality_df_11C.to_csv(
    QUALITY_OUTPUT_PATH_11C,
    index=False
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print()
print("QUALITY SUMMARY")
print("-" * 70)

summary_cols_11C = [
    "mean_lead_rms_mV",
    "mean_lead_range_mV",
    "mean_baseline_wander_ratio",
    "mean_high_frequency_ratio",
    "zero_rms_leads"
]

print(
    quality_df_11C[summary_cols_11C]
    .describe()
    .T
    .to_string()
)

print()
print("ECGs containing at least one zero-RMS lead:",
      int((quality_df_11C["zero_rms_leads"] > 0).sum()))

print()
print("Saved:")
print(QUALITY_OUTPUT_PATH_11C)

print()
print("=" * 70)
print("STEP 11C STATUS: PASS")
print("=" * 70)

STEP 11C — NATURAL PTB-XL SIGNAL-QUALITY CHARACTERIZATION

Test ECGs: 2198
Leads per ECG: 12
Sampling frequency: 500 Hz

Processed  100/2198 ECGs
Processed  200/2198 ECGs
Processed  300/2198 ECGs
Processed  400/2198 ECGs
Processed  500/2198 ECGs
Processed  600/2198 ECGs
Processed  700/2198 ECGs
Processed  800/2198 ECGs
Processed  900/2198 ECGs
Processed 1000/2198 ECGs
Processed 1100/2198 ECGs
Processed 1200/2198 ECGs
Processed 1300/2198 ECGs
Processed 1400/2198 ECGs
Processed 1500/2198 ECGs
Processed 1600/2198 ECGs
Processed 1700/2198 ECGs
Processed 1800/2198 ECGs
Processed 1900/2198 ECGs
Processed 2000/2198 ECGs
Processed 2100/2198 ECGs
Processed 2198/2198 ECGs

QUALITY SUMMARY
----------------------------------------------------------------------
                             count      mean       std       min       25%       50%       75%       max
mean_lead_rms_mV            2198.0  0.191988  0.072958  0.073396  0.147262  0.174757  0.215218  0.787280
mean_lead_range_mV          219

In [7]:
import os
import json
import numpy as np

# ============================================================
# STEP 11D — CONTROLLED NOISE CONSTRUCTION VALIDATION
# ============================================================

NOISE_RESULTS_PATH_11D = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "results/noise_robustness"
)

PROTOCOL_PATH_11D = os.path.join(
    NOISE_RESULTS_PATH_11D,
    "step11B_noise_robustness_protocol.json"
)

with open(PROTOCOL_PATH_11D, "r", encoding="utf-8") as f:
    protocol_11D = json.load(f)

SEED_11D = protocol_11D["random_seed"]

SNR_LEVELS_11D = [
    float(x)
    for x in protocol_11D["noise_levels_db"]
]

EXPECTED_SHAPE_11D = (12, 5000)


def add_gaussian_noise_at_snr_11D(
    x,
    snr_db,
    rng
):
    """
    Add zero-mean Gaussian noise whose power is chosen
    to achieve the requested signal-to-noise ratio.

    SNR(dB) = 10 * log10(signal_power / noise_power)

    x shape:
        (12, 5000)
    """

    x = np.asarray(x, dtype=np.float32)

    if x.shape != EXPECTED_SHAPE_11D:
        raise ValueError(
            f"Expected {EXPECTED_SHAPE_11D}, got {x.shape}"
        )

    if not np.isfinite(x).all():
        raise ValueError(
            "Input contains non-finite values."
        )

    signal_power = float(
        np.mean(
            np.square(
                x.astype(np.float64)
            )
        )
    )

    if signal_power <= 0:
        raise ValueError(
            "Signal power must be positive."
        )

    noise_power = (
        signal_power /
        (10.0 ** (snr_db / 10.0))
    )

    noise_std = np.sqrt(noise_power)

    noise = rng.normal(
        loc=0.0,
        scale=noise_std,
        size=x.shape
    ).astype(np.float32)

    noisy = x + noise

    return noisy


def measured_snr_db_11D(
    clean,
    noisy
):
    """
    Measure the achieved SNR directly from the generated
    perturbation.
    """

    clean = np.asarray(
        clean,
        dtype=np.float64
    )

    noisy = np.asarray(
        noisy,
        dtype=np.float64
    )

    noise = noisy - clean

    signal_power = np.mean(
        np.square(clean)
    )

    noise_power = np.mean(
        np.square(noise)
    )

    return float(
        10.0 *
        np.log10(
            signal_power /
            noise_power
        )
    )


# ------------------------------------------------------------
# Reproducibility and SNR validation
# ------------------------------------------------------------

rng_11D = np.random.default_rng(SEED_11D)

# Synthetic but deterministic validation waveform.
# This validates the noise-generation mathematics without
# touching the locked test set.
t_11D = np.arange(
    EXPECTED_SHAPE_11D[1],
    dtype=np.float32
) / 500.0

test_signal_11D = np.zeros(
    EXPECTED_SHAPE_11D,
    dtype=np.float32
)

for lead_idx in range(12):
    test_signal_11D[lead_idx] = (
        0.8 * np.sin(
            2.0 * np.pi * 1.2 * t_11D
        )
        + 0.2 * np.sin(
            2.0 * np.pi * 8.0 * t_11D
        )
    )

validation_rows_11D = []

for snr_db in SNR_LEVELS_11D:

    noisy_1_11D = add_gaussian_noise_at_snr_11D(
        test_signal_11D,
        snr_db,
        rng_11D
    )

    measured_1_11D = measured_snr_db_11D(
        test_signal_11D,
        noisy_1_11D
    )

    # Reset RNG and regenerate to verify reproducibility.
    rng_repeat_11D = np.random.default_rng(SEED_11D)

    # Advance the RNG identically through previous conditions.
    for previous_snr_11D in SNR_LEVELS_11D[
        :SNR_LEVELS_11D.index(snr_db)
    ]:
        _ = add_gaussian_noise_at_snr_11D(
            test_signal_11D,
            previous_snr_11D,
            rng_repeat_11D
        )

    noisy_2_11D = add_gaussian_noise_at_snr_11D(
        test_signal_11D,
        snr_db,
        rng_repeat_11D
    )

    reproducible_11D = np.array_equal(
        noisy_1_11D,
        noisy_2_11D
    )

    measured_2_11D = measured_snr_db_11D(
        test_signal_11D,
        noisy_2_11D
    )

    snr_error_11D = abs(
        measured_1_11D - snr_db
    )

    validation_rows_11D.append({
        "snr_db_requested": snr_db,
        "snr_db_measured": measured_1_11D,
        "snr_absolute_error_db": snr_error_11D,
        "reproducible": reproducible_11D,
        "finite_output": bool(
            np.isfinite(noisy_1_11D).all()
        ),
        "shape_correct": (
            noisy_1_11D.shape ==
            EXPECTED_SHAPE_11D
        )
    })

validation_df_11D = (
    __import__("pandas")
    .DataFrame(validation_rows_11D)
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(validation_df_11D) == len(
    SNR_LEVELS_11D
)

assert validation_df_11D[
    "finite_output"
].all()

assert validation_df_11D[
    "shape_correct"
].all()

assert validation_df_11D[
    "reproducible"
].all()

assert (
    validation_df_11D[
        "snr_absolute_error_db"
    ].max()
    < 0.1
)

# ------------------------------------------------------------
# Save validation record
# ------------------------------------------------------------

OUTPUT_PATH_11D = os.path.join(
    NOISE_RESULTS_PATH_11D,
    "step11D_noise_generation_validation.csv"
)

validation_df_11D.to_csv(
    OUTPUT_PATH_11D,
    index=False
)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 70)
print("STEP 11D — CONTROLLED NOISE CONSTRUCTION VALIDATION")
print("=" * 70)
print()

print("Noise type: Additive zero-mean Gaussian")
print("Random seed:", SEED_11D)
print("Input shape:", EXPECTED_SHAPE_11D)
print()

print("SNR VALIDATION")
print("-" * 70)

print(
    validation_df_11D.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print()
print(
    "Maximum absolute SNR error:",
    f'{validation_df_11D["snr_absolute_error_db"].max():.6f} dB'
)

print(
    "All outputs finite:",
    validation_df_11D["finite_output"].all()
)

print(
    "All outputs correct shape:",
    validation_df_11D["shape_correct"].all()
)

print(
    "Noise generation reproducible:",
    validation_df_11D["reproducible"].all()
)

print()
print("Saved:")
print(OUTPUT_PATH_11D)

print()
print("=" * 70)
print("STEP 11D STATUS: PASS")
print("=" * 70)

STEP 11D — CONTROLLED NOISE CONSTRUCTION VALIDATION

Noise type: Additive zero-mean Gaussian
Random seed: 42
Input shape: (12, 5000)

SNR VALIDATION
----------------------------------------------------------------------
 snr_db_requested  snr_db_measured  snr_absolute_error_db  reproducible  finite_output  shape_correct
        30.000000        29.981005               0.018995          True           True           True
        20.000000        19.953925               0.046075          True           True           True
        10.000000         9.989955               0.010045          True           True           True
         0.000000         0.029615               0.029615          True           True           True

Maximum absolute SNR error: 0.046075 dB
All outputs finite: True
All outputs correct shape: True
Noise generation reproducible: True

Saved:
/content/drive/MyDrive/PTB-XL Research Project/results/noise_robustness/step11D_noise_generation_validation.csv

STEP 11D STATUS

In [10]:
# ============================================================
# STEP 11E — MODEL ARCHITECTURE CORRECTION / CHECKPOINT VERIFY
# ============================================================

import torch
import torch.nn as nn
import os

CHECKPOINT_PATH_11E = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "results/deep_learning/ecg_resnet1d_best_validation.pt"
)

device_11E = torch.device("cpu")


# ------------------------------------------------------------
# EXACT NOTEBOOK 10 NAMING / ARCHITECTURE
# ------------------------------------------------------------

class ResidualBlock1D_11E(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=7,
            stride=stride,
            padding=3,
            bias=False
        )

        self.bn1 = nn.BatchNorm1d(
            out_channels
        )

        self.relu = nn.ReLU(
            inplace=True
        )

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=7,
            stride=1,
            padding=3,
            bias=False
        )

        self.bn2 = nn.BatchNorm1d(
            out_channels
        )

        # IMPORTANT:
        # Notebook 10 uses the name "shortcut"
        # rather than "downsample".
        if (
            stride != 1
            or in_channels != out_channels
        ):

            self.shortcut = nn.Sequential(
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm1d(
                    out_channels
                )
            )

        else:

            self.shortcut = nn.Identity()

    def forward(self, x):

        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)

        return out


class ECGResNet1D_11E(nn.Module):

    def __init__(
        self,
        n_leads=12,
        n_classes=5
    ):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(
                n_leads,
                64,
                kernel_size=15,
                stride=2,
                padding=7,
                bias=False
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(
                kernel_size=3,
                stride=2,
                padding=1
            )
        )

        self.layer1 = nn.Sequential(
            ResidualBlock1D_11E(
                64,
                64
            ),
            ResidualBlock1D_11E(
                64,
                64
            )
        )

        self.layer2 = nn.Sequential(
            ResidualBlock1D_11E(
                64,
                128,
                stride=2
            ),
            ResidualBlock1D_11E(
                128,
                128
            )
        )

        self.layer3 = nn.Sequential(
            ResidualBlock1D_11E(
                128,
                256,
                stride=2
            ),
            ResidualBlock1D_11E(
                256,
                256
            )
        )

        self.layer4 = nn.Sequential(
            ResidualBlock1D_11E(
                256,
                512,
                stride=2
            ),
            ResidualBlock1D_11E(
                512,
                512
            )
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        # IMPORTANT:
        # Notebook 10 uses "classifier", not "fc".
        self.classifier = nn.Linear(
            512,
            n_classes
        )

    def forward(self, x):

        x = self.stem(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.global_pool(x)

        x = x.squeeze(-1)

        return self.classifier(x)


# ------------------------------------------------------------
# LOAD CHECKPOINT
# ------------------------------------------------------------

if not os.path.exists(
    CHECKPOINT_PATH_11E
):

    raise FileNotFoundError(
        CHECKPOINT_PATH_11E
    )

checkpoint_11E = torch.load(
    CHECKPOINT_PATH_11E,
    map_location=device_11E,
    weights_only=False
)

model_11E = ECGResNet1D_11E(
    n_leads=12,
    n_classes=5
)

# ------------------------------------------------------------
# STRICT CHECKPOINT LOAD
# ------------------------------------------------------------

model_11E.load_state_dict(
    checkpoint_11E["model_state_dict"],
    strict=True
)

model_11E.to(device_11E)
model_11E.eval()


# ------------------------------------------------------------
# VERIFY MODEL SIZE / CHECKPOINT
# ------------------------------------------------------------

parameter_count_11E = sum(
    p.numel()
    for p in model_11E.parameters()
)

assert parameter_count_11E == 8_739_973
assert checkpoint_11E["epoch"] == 17


# ------------------------------------------------------------
# VERIFY A REAL FORWARD PASS
# ------------------------------------------------------------

with torch.no_grad():

    test_input_11E = torch.zeros(
        1,
        12,
        5000,
        dtype=torch.float32
    )

    test_output_11E = model_11E(
        test_input_11E
    )

assert test_output_11E.shape == (
    1,
    5
)

assert torch.isfinite(
    test_output_11E
).all()


print("=" * 70)
print("STEP 11E — CHECKPOINT VERIFICATION")
print("=" * 70)
print()
print("Checkpoint loaded successfully with strict=True.")
print("Missing keys: 0")
print("Unexpected keys: 0")
print()
print("Architecture: ECGResNet1D")
print("Parameters:", f"{parameter_count_11E:,}")
print("Checkpoint epoch:", checkpoint_11E["epoch"])
print(
    "Best validation macro-AUROC:",
    f"{checkpoint_11E['best_validation_macro_auroc']:.6f}"
)
print("Test input shape:", tuple(test_input_11E.shape))
print("Test output shape:", tuple(test_output_11E.shape))
print()
print("=" * 70)
print("MODEL VERIFICATION: PASS")
print("=" * 70)

STEP 11E — CHECKPOINT VERIFICATION

Checkpoint loaded successfully with strict=True.
Missing keys: 0
Unexpected keys: 0

Architecture: ECGResNet1D
Parameters: 8,739,973
Checkpoint epoch: 17
Best validation macro-AUROC: 0.934005
Test input shape: (1, 12, 5000)
Test output shape: (1, 5)

MODEL VERIFICATION: PASS


In [12]:
# ============================================================
# STEP 11E — PTB-XL PATH VERIFICATION
# ============================================================

import os
import pandas as pd
import wfdb

PROJECT_PATH_11E = (
    "/content/drive/MyDrive/PTB-XL Research Project"
)

DATA_PATH_11E = (
    PROJECT_PATH_11E +
    "/data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
)

DATABASE_PATH_11E = os.path.join(
    DATA_PATH_11E,
    "ptbxl_database.csv"
)

SPLIT_PATH_11E = os.path.join(
    PROJECT_PATH_11E,
    "results",
    "ptbxl_patient_independent_split.csv"
)

print("=" * 70)
print("STEP 11E — PTB-XL PATH VERIFICATION")
print("=" * 70)
print()

# ------------------------------------------------------------
# Verify database
# ------------------------------------------------------------

if not os.path.exists(DATABASE_PATH_11E):

    raise FileNotFoundError(
        f"ptbxl_database.csv not found:\n{DATABASE_PATH_11E}"
    )

db_11E = pd.read_csv(
    DATABASE_PATH_11E
)

print("Database loaded.")
print("Rows:", len(db_11E))
print("Columns containing filename:")
print(
    [
        c for c in db_11E.columns
        if "filename" in c.lower()
    ]
)

# ------------------------------------------------------------
# Locate failing ECG from previous error
# ------------------------------------------------------------

TEST_ECG_ID_11E = 1009

row_11E = db_11E[
    db_11E["ecg_id"] == TEST_ECG_ID_11E
]

if len(row_11E) != 1:

    raise RuntimeError(
        f"Expected exactly one row for ECG "
        f"{TEST_ECG_ID_11E}, found {len(row_11E)}."
    )

row_11E = row_11E.iloc[0]

print()
print("ECG ID:", TEST_ECG_ID_11E)

# ------------------------------------------------------------
# Show canonical filenames
# ------------------------------------------------------------

filename_columns_11E = [
    c for c in db_11E.columns
    if "filename" in c.lower()
]

for column in filename_columns_11E:

    print(
        f"{column}:",
        row_11E[column]
    )

# ------------------------------------------------------------
# Test filename_hr directly
# ------------------------------------------------------------

if "filename_hr" not in db_11E.columns:

    raise RuntimeError(
        "PTB-XL database does not contain "
        "'filename_hr'."
    )

filename_hr_11E = str(
    row_11E["filename_hr"]
)

print()
print("Canonical filename_hr:")
print(filename_hr_11E)

# ------------------------------------------------------------
# Construct canonical path
# ------------------------------------------------------------

canonical_path_11E = os.path.join(
    DATA_PATH_11E,
    filename_hr_11E
)

print()
print("Canonical filesystem path:")
print(canonical_path_11E)

print()
print("Header exists:")
print(
    os.path.exists(
        canonical_path_11E + ".hea"
    )
)

print("Signal file exists:")
print(
    os.path.exists(
        canonical_path_11E + ".dat"
    )
)

# ------------------------------------------------------------
# Test actual WFDB read
# ------------------------------------------------------------

if not os.path.exists(
    canonical_path_11E + ".hea"
):

    raise FileNotFoundError(
        "Canonical filename_hr header does not exist:\n"
        + canonical_path_11E + ".hea"
    )

signal_11E, fields_11E = wfdb.rdsamp(
    canonical_path_11E
)

print()
print("WFDB read: SUCCESS")
print("Signal shape:", signal_11E.shape)
print("Sampling frequency:", fields_11E["fs"])
print("Number of channels:", signal_11E.shape[1])

assert signal_11E.shape == (
    5000,
    12
)

assert fields_11E["fs"] == 500

print()
print("=" * 70)
print("PATH VERIFICATION: PASS")
print("=" * 70)

STEP 11E — PTB-XL PATH VERIFICATION

Database loaded.
Rows: 21799
Columns containing filename:
['filename_lr', 'filename_hr']

ECG ID: 1009
filename_lr: records100/01000/01009_lr
filename_hr: records500/01000/01009_hr

Canonical filename_hr:
records500/01000/01009_hr

Canonical filesystem path:
/content/drive/MyDrive/PTB-XL Research Project/data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/records500/01000/01009_hr

Header exists:
True
Signal file exists:
True

WFDB read: SUCCESS
Signal shape: (5000, 12)
Sampling frequency: 500
Number of channels: 12

PATH VERIFICATION: PASS


In [13]:
# ============================================================
# STEP 11E — FROZEN RESNET NOISE-ROBUSTNESS EVALUATION
# FINAL VERIFIED VERSION
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import torch
import wfdb

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    recall_score,
    confusion_matrix
)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_PATH_11E = (
    "/content/drive/MyDrive/PTB-XL Research Project"
)

DATA_PATH_11E = (
    PROJECT_PATH_11E +
    "/data/ptb-xl-a-large-publicly-available-"
    "electrocardiography-dataset-1.0.3"
)

RESULTS_PATH_11E = (
    PROJECT_PATH_11E +
    "/results"
)

NOISE_RESULTS_PATH_11E = (
    RESULTS_PATH_11E +
    "/noise_robustness"
)

DATABASE_PATH_11E = os.path.join(
    DATA_PATH_11E,
    "ptbxl_database.csv"
)

SPLIT_PATH_11E = os.path.join(
    RESULTS_PATH_11E,
    "ptbxl_patient_independent_split.csv"
)

TARGETS_PATH_11E = os.path.join(
    RESULTS_PATH_11E,
    "ptbxl_diagnostic_targets.csv"
)

OUTPUT_PATH_11E = os.path.join(
    NOISE_RESULTS_PATH_11E,
    "step11E_noise_robustness_results.csv"
)

os.makedirs(
    NOISE_RESULTS_PATH_11E,
    exist_ok=True
)

# ------------------------------------------------------------
# FROZEN EXPERIMENT SETTINGS
# ------------------------------------------------------------

TARGET_COLUMNS_11E = [
    "MI",
    "STTC",
    "CD",
    "HYP",
    "NORM"
]

NOISE_CONDITIONS_11E = [
    ("clean", None),
    ("snr_30db", 30.0),
    ("snr_20db", 20.0),
    ("snr_10db", 10.0),
    ("snr_0db", 0.0)
]

SEED_11E = 42

EXPECTED_TEST_N_11E = 2198
EXPECTED_PARAMETER_COUNT_11E = 8_739_973
EXPECTED_EPOCH_11E = 17
EXPECTED_CLEAN_AUROC_11E = 0.930060

# ------------------------------------------------------------
# VERIFY VERIFIED MODEL
# ------------------------------------------------------------

if "model_11E" not in globals():

    raise RuntimeError(
        "Verified model_11E is not available. "
        "Run the checkpoint-verification cell first."
    )

model_11E.eval()

parameter_count_11E = sum(
    p.numel()
    for p in model_11E.parameters()
)

assert (
    parameter_count_11E
    == EXPECTED_PARAMETER_COUNT_11E
)

# ------------------------------------------------------------
# LOAD PTB-XL DATABASE
# ------------------------------------------------------------

db_11E = pd.read_csv(
    DATABASE_PATH_11E
)

assert len(db_11E) == 21799
assert "ecg_id" in db_11E.columns
assert "filename_hr" in db_11E.columns

db_11E["ecg_id"] = (
    db_11E["ecg_id"].astype(int)
)

db_11E = db_11E.set_index(
    "ecg_id"
)

# ------------------------------------------------------------
# LOAD LOCKED SPLIT
# ------------------------------------------------------------

split_df_11E = pd.read_csv(
    SPLIT_PATH_11E
)

test_ids_11E = (
    split_df_11E.loc[
        split_df_11E["split"] == "test",
        "ecg_id"
    ]
    .astype(int)
    .tolist()
)

assert len(test_ids_11E) == (
    EXPECTED_TEST_N_11E
)

assert len(set(test_ids_11E)) == (
    EXPECTED_TEST_N_11E
)

# ------------------------------------------------------------
# LOAD TARGETS
# ------------------------------------------------------------

targets_df_11E = pd.read_csv(
    TARGETS_PATH_11E
)

targets_df_11E["ecg_id"] = (
    targets_df_11E["ecg_id"]
    .astype(int)
)

test_targets_11E = (
    targets_df_11E[
        targets_df_11E["ecg_id"].isin(
            test_ids_11E
        )
    ]
    .set_index("ecg_id")
    .loc[test_ids_11E]
    .reset_index()
)

assert len(test_targets_11E) == (
    EXPECTED_TEST_N_11E
)

Y_test_11E = (
    test_targets_11E[
        TARGET_COLUMNS_11E
    ]
    .to_numpy(
        dtype=np.float32
    )
)

assert Y_test_11E.shape == (
    EXPECTED_TEST_N_11E,
    5
)

# ------------------------------------------------------------
# CANONICAL WFDB LOADER
# ------------------------------------------------------------

def load_ecg_11E(ecg_id):

    if ecg_id not in db_11E.index:

        raise KeyError(
            f"ECG {ecg_id} not found in "
            "PTB-XL database."
        )

    filename_hr = str(
        db_11E.loc[
            ecg_id,
            "filename_hr"
        ]
    )

    record_path = os.path.join(
        DATA_PATH_11E,
        filename_hr
    )

    if not os.path.exists(
        record_path + ".hea"
    ):

        raise FileNotFoundError(
            "Missing PTB-XL header:\n"
            + record_path + ".hea"
        )

    signal, fields = wfdb.rdsamp(
        record_path
    )

    signal = np.asarray(
        signal,
        dtype=np.float32
    )

    assert signal.shape == (
        5000,
        12
    )

    assert fields["fs"] == 500

    signal = signal.T

    assert signal.shape == (
        12,
        5000
    )

    assert np.isfinite(
        signal
    ).all()

    return signal


# ------------------------------------------------------------
# GAUSSIAN NOISE
# ------------------------------------------------------------

def add_noise_11E(
    x,
    snr_db,
    rng
):

    if snr_db is None:

        return x.copy()

    signal_power = np.mean(
        np.square(
            x.astype(np.float64)
        )
    )

    if signal_power <= 0:

        raise ValueError(
            "Signal power must be positive."
        )

    noise_power = (
        signal_power
        / (10.0 ** (snr_db / 10.0))
    )

    noise_std = np.sqrt(
        noise_power
    )

    noise = rng.normal(
        loc=0.0,
        scale=noise_std,
        size=x.shape
    ).astype(np.float32)

    return x + noise


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

def calculate_metrics_11E(
    y_true,
    probabilities
):

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    rows = []

    for class_idx, target in enumerate(
        TARGET_COLUMNS_11E
    ):

        y = y_true[:, class_idx]
        p = probabilities[:, class_idx]
        pred = predictions[:, class_idx]

        auroc = roc_auc_score(
            y,
            p
        )

        auprc = average_precision_score(
            y,
            p
        )

        f1 = f1_score(
            y,
            pred,
            zero_division=0
        )

        sensitivity = recall_score(
            y,
            pred,
            zero_division=0
        )

        tn, fp, fn, tp = confusion_matrix(
            y,
            pred,
            labels=[0, 1]
        ).ravel()

        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        )

        rows.append({
            "target": target,
            "AUROC": auroc,
            "AUPRC": auprc,
            "F1": f1,
            "sensitivity": sensitivity,
            "specificity": specificity
        })

    class_df = pd.DataFrame(
        rows
    )

    macro_row = {
        "target": "MACRO",
        "AUROC": class_df["AUROC"].mean(),
        "AUPRC": class_df["AUPRC"].mean(),
        "F1": class_df["F1"].mean(),
        "sensitivity": class_df["sensitivity"].mean(),
        "specificity": class_df["specificity"].mean()
    }

    return pd.concat(
        [
            class_df,
            pd.DataFrame([macro_row])
        ],
        ignore_index=True
    )


# ------------------------------------------------------------
# EVALUATION
# ------------------------------------------------------------

all_results_11E = []

total_start_11E = time.time()

print("=" * 70)
print("STEP 11E — FROZEN RESNET NOISE-ROBUSTNESS EVALUATION")
print("=" * 70)
print()
print("Test ECGs:", len(test_ids_11E))
print(
    "Model parameters:",
    f"{parameter_count_11E:,}"
)
print("Checkpoint epoch:", EXPECTED_EPOCH_11E)
print("Device:", next(model_11E.parameters()).device)
print("Noise type: additive zero-mean Gaussian")
print("Threshold for classification metrics: 0.5")
print("Test set: locked patient-independent test set")
print()

for condition_idx, (
    condition,
    snr_db
) in enumerate(
    NOISE_CONDITIONS_11E
):

    print("=" * 70)
    print(
        f"CONDITION {condition_idx + 1}/5: "
        f"{condition}"
    )
    print("=" * 70)

    condition_start_11E = time.time()

    rng_11E = np.random.default_rng(
        SEED_11E + condition_idx
    )

    probabilities_11E = []

    with torch.no_grad():

        for i, ecg_id in enumerate(
            test_ids_11E,
            start=1
        ):

            x = load_ecg_11E(
                ecg_id
            )

            x_input = add_noise_11E(
                x,
                snr_db,
                rng_11E
            )

            tensor = torch.from_numpy(
                x_input
            ).unsqueeze(0)

            logits = model_11E(
                tensor
            )

            probabilities = (
                torch.sigmoid(
                    logits
                )
                .cpu()
                .numpy()[0]
            )

            probabilities_11E.append(
                probabilities
            )

            if (
                i % 250 == 0
                or i == EXPECTED_TEST_N_11E
            ):

                elapsed = (
                    time.time()
                    - condition_start_11E
                )

                rate = (
                    i / elapsed
                    if elapsed > 0
                    else 0
                )

                print(
                    f"Processed {i:4d}/"
                    f"{EXPECTED_TEST_N_11E} "
                    f"| {rate:.2f} ECG/s "
                    f"| elapsed "
                    f"{elapsed / 60:.2f} min"
                )

    probabilities_11E = np.asarray(
        probabilities_11E,
        dtype=np.float64
    )

    assert probabilities_11E.shape == (
        EXPECTED_TEST_N_11E,
        5
    )

    assert np.isfinite(
        probabilities_11E
    ).all()

    metrics_11E = calculate_metrics_11E(
        Y_test_11E,
        probabilities_11E
    )

    condition_time_11E = (
        time.time()
        - condition_start_11E
    )

    print()
    print(
        metrics_11E.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

    for _, row in metrics_11E.iterrows():

        all_results_11E.append({
            "condition": condition,
            "snr_db": snr_db,
            "target": row["target"],
            "AUROC": row["AUROC"],
            "AUPRC": row["AUPRC"],
            "F1": row["F1"],
            "sensitivity": row["sensitivity"],
            "specificity": row["specificity"],
            "evaluation_time_seconds":
                condition_time_11E
        })


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

results_df_11E = pd.DataFrame(
    all_results_11E
)

assert len(results_df_11E) == 30

# ------------------------------------------------------------
# CLEAN INTEGRITY CHECK
# ------------------------------------------------------------

clean_macro_11E = (
    results_df_11E[
        (
            results_df_11E["condition"]
            == "clean"
        )
        &
        (
            results_df_11E["target"]
            == "MACRO"
        )
    ]
)

assert len(clean_macro_11E) == 1

clean_auroc_11E = float(
    clean_macro_11E["AUROC"].iloc[0]
)

clean_difference_11E = abs(
    clean_auroc_11E
    - EXPECTED_CLEAN_AUROC_11E
)

print()
print("=" * 70)
print("CLEAN-CONDITION INTEGRITY CHECK")
print("=" * 70)
print()
print(
    "Notebook 10 frozen test macro-AUROC:",
    f"{EXPECTED_CLEAN_AUROC_11E:.6f}"
)

print(
    "Notebook 11 clean macro-AUROC:",
    f"{clean_auroc_11E:.6f}"
)

print(
    "Absolute difference:",
    f"{clean_difference_11E:.6f}"
)

if clean_difference_11E > 0.005:

    raise RuntimeError(
        "CLEAN-CONDITION INTEGRITY CHECK FAILED. "
        f"Difference = {clean_difference_11E:.6f}. "
        "Do not interpret the noise results."
    )

print()
print(
    "CLEAN-CONDITION INTEGRITY CHECK: PASS"
)

# ------------------------------------------------------------
# DEGRADATION FROM CLEAN
# ------------------------------------------------------------

clean_values_11E = (
    results_df_11E[
        results_df_11E["condition"] == "clean"
    ]
    .set_index("target")
)

results_df_11E[
    "AUROC_degradation_from_clean"
] = results_df_11E.apply(
    lambda row:
        float(
            clean_values_11E.loc[
                row["target"],
                "AUROC"
            ]
        )
        - float(row["AUROC"]),
    axis=1
)

results_df_11E[
    "AUPRC_degradation_from_clean"
] = results_df_11E.apply(
    lambda row:
        float(
            clean_values_11E.loc[
                row["target"],
                "AUPRC"
            ]
        )
        - float(row["AUPRC"]),
    axis=1
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

results_df_11E.to_csv(
    OUTPUT_PATH_11E,
    index=False
)

total_time_11E = (
    time.time()
    - total_start_11E
)

# ------------------------------------------------------------
# FINAL MACRO SUMMARY
# ------------------------------------------------------------

macro_summary_11E = (
    results_df_11E[
        results_df_11E["target"] == "MACRO"
    ][
        [
            "condition",
            "snr_db",
            "AUROC",
            "AUPRC",
            "F1",
            "sensitivity",
            "specificity",
            "AUROC_degradation_from_clean",
            "AUPRC_degradation_from_clean"
        ]
    ]
)

print()
print("=" * 70)
print("STEP 11E — MACRO ROBUSTNESS RESULTS")
print("=" * 70)
print()

print(
    macro_summary_11E.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print()
print(
    "Total evaluation time:",
    f"{total_time_11E / 60:.2f} minutes"
)

print()
print("Results saved to:")
print(OUTPUT_PATH_11E)

print()
print("=" * 70)
print("STEP 11E STATUS: PASS")
print("=" * 70)

STEP 11E — FROZEN RESNET NOISE-ROBUSTNESS EVALUATION

Test ECGs: 2198
Model parameters: 8,739,973
Checkpoint epoch: 17
Device: cpu
Noise type: additive zero-mean Gaussian
Threshold for classification metrics: 0.5
Test set: locked patient-independent test set

CONDITION 1/5: clean
Processed  250/2198 | 7.84 ECG/s | elapsed 0.53 min
Processed  500/2198 | 9.16 ECG/s | elapsed 0.91 min
Processed  750/2198 | 9.89 ECG/s | elapsed 1.26 min
Processed 1000/2198 | 10.16 ECG/s | elapsed 1.64 min
Processed 1250/2198 | 10.46 ECG/s | elapsed 1.99 min
Processed 1500/2198 | 10.49 ECG/s | elapsed 2.38 min
Processed 1750/2198 | 10.67 ECG/s | elapsed 2.73 min
Processed 2000/2198 | 10.73 ECG/s | elapsed 3.11 min
Processed 2198/2198 | 10.81 ECG/s | elapsed 3.39 min

target    AUROC    AUPRC       F1  sensitivity  specificity
    MI 0.934836 0.850112 0.736428     0.850909     0.846481
  STTC 0.933429 0.823583 0.734177     0.890595     0.833631
    CD 0.924570 0.843310 0.726957     0.842742     0.861340
   H

In [14]:
# ============================================================
# STEP 11F — PER-CLASS ROBUSTNESS ANALYSIS
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_PATH_11F = (
    "/content/drive/MyDrive/PTB-XL Research Project"
)

NOISE_RESULTS_PATH_11F = os.path.join(
    PROJECT_PATH_11F,
    "results",
    "noise_robustness"
)

INPUT_PATH_11F = os.path.join(
    NOISE_RESULTS_PATH_11F,
    "step11E_noise_robustness_results.csv"
)

OUTPUT_PATH_11F = os.path.join(
    NOISE_RESULTS_PATH_11F,
    "step11F_per_class_robustness.csv"
)

# ------------------------------------------------------------
# LOAD FROZEN 11E RESULTS
# ------------------------------------------------------------

assert os.path.exists(INPUT_PATH_11F), (
    f"11E results not found:\n{INPUT_PATH_11F}"
)

df_11F = pd.read_csv(
    INPUT_PATH_11F
)

EXPECTED_CLASSES_11F = [
    "MI",
    "STTC",
    "CD",
    "HYP",
    "NORM"
]

EXPECTED_CONDITIONS_11F = [
    "clean",
    "snr_30db",
    "snr_20db",
    "snr_10db",
    "snr_0db"
]

# ------------------------------------------------------------
# BASIC INTEGRITY CHECKS
# ------------------------------------------------------------

assert set(
    df_11F["target"].unique()
) == set(
    EXPECTED_CLASSES_11F + ["MACRO"]
)

assert set(
    df_11F["condition"].unique()
) == set(
    EXPECTED_CONDITIONS_11F
)

assert len(df_11F) == 30

for column in [
    "AUROC",
    "AUPRC",
    "F1",
    "sensitivity",
    "specificity"
]:

    assert np.isfinite(
        df_11F[column].to_numpy()
    ).all()

# ------------------------------------------------------------
# EXCLUDE MACRO FOR PER-CLASS ANALYSIS
# ------------------------------------------------------------

class_df_11F = df_11F[
    df_11F["target"].isin(
        EXPECTED_CLASSES_11F
    )
].copy()

assert len(class_df_11F) == 25

# ------------------------------------------------------------
# CREATE WIDE AUROC TABLE
# ------------------------------------------------------------

auroc_table_11F = (
    class_df_11F
    .pivot(
        index="target",
        columns="condition",
        values="AUROC"
    )
    .loc[
        EXPECTED_CLASSES_11F,
        EXPECTED_CONDITIONS_11F
    ]
)

# ------------------------------------------------------------
# CREATE WIDE AUPRC TABLE
# ------------------------------------------------------------

auprc_table_11F = (
    class_df_11F
    .pivot(
        index="target",
        columns="condition",
        values="AUPRC"
    )
    .loc[
        EXPECTED_CLASSES_11F,
        EXPECTED_CONDITIONS_11F
    ]
)

# ------------------------------------------------------------
# CREATE WIDE F1 TABLE
# ------------------------------------------------------------

f1_table_11F = (
    class_df_11F
    .pivot(
        index="target",
        columns="condition",
        values="F1"
    )
    .loc[
        EXPECTED_CLASSES_11F,
        EXPECTED_CONDITIONS_11F
    ]
)

# ------------------------------------------------------------
# CALCULATE DEGRADATION FROM CLEAN
# ------------------------------------------------------------

clean_class_11F = (
    class_df_11F[
        class_df_11F["condition"] == "clean"
    ]
    .set_index("target")
)

class_df_11F[
    "AUROC_drop_from_clean"
] = class_df_11F.apply(
    lambda row:
        clean_class_11F.loc[
            row["target"],
            "AUROC"
        ] - row["AUROC"],
    axis=1
)

class_df_11F[
    "AUPRC_drop_from_clean"
] = class_df_11F.apply(
    lambda row:
        clean_class_11F.loc[
            row["target"],
            "AUPRC"
        ] - row["AUPRC"],
    axis=1
)

class_df_11F[
    "F1_drop_from_clean"
] = class_df_11F.apply(
    lambda row:
        clean_class_11F.loc[
            row["target"],
            "F1"
        ] - row["F1"],
    axis=1
)

class_df_11F[
    "sensitivity_change_from_clean"
] = class_df_11F.apply(
    lambda row:
        row["sensitivity"]
        - clean_class_11F.loc[
            row["target"],
            "sensitivity"
        ],
    axis=1
)

class_df_11F[
    "specificity_change_from_clean"
] = class_df_11F.apply(
    lambda row:
        row["specificity"]
        - clean_class_11F.loc[
            row["target"],
            "specificity"
        ],
    axis=1
)

# ------------------------------------------------------------
# 0 dB SUMMARY
# ------------------------------------------------------------

zero_db_11F = (
    class_df_11F[
        class_df_11F["condition"] == "snr_0db"
    ]
    .copy()
)

zero_db_11F = zero_db_11F[
    [
        "target",
        "AUROC",
        "AUPRC",
        "F1",
        "sensitivity",
        "specificity",
        "AUROC_drop_from_clean",
        "AUPRC_drop_from_clean",
        "F1_drop_from_clean",
        "sensitivity_change_from_clean",
        "specificity_change_from_clean"
    ]
]

zero_db_11F = zero_db_11F.sort_values(
    "AUROC_drop_from_clean",
    ascending=False
)

# ------------------------------------------------------------
# FIND MOST SENSITIVE CLASS
# ------------------------------------------------------------

largest_auroc_drop_row_11F = (
    zero_db_11F.iloc[0]
)

largest_auroc_drop_class_11F = (
    largest_auroc_drop_row_11F["target"]
)

largest_auroc_drop_11F = float(
    largest_auroc_drop_row_11F[
        "AUROC_drop_from_clean"
    ]
)

largest_auprc_drop_row_11F = (
    zero_db_11F.sort_values(
        "AUPRC_drop_from_clean",
        ascending=False
    ).iloc[0]
)

largest_auprc_drop_class_11F = (
    largest_auprc_drop_row_11F["target"]
)

largest_auprc_drop_11F = float(
    largest_auprc_drop_row_11F[
        "AUPRC_drop_from_clean"
    ]
)

# ------------------------------------------------------------
# SAVE FULL PER-CLASS TABLE
# ------------------------------------------------------------

class_df_11F.to_csv(
    OUTPUT_PATH_11F,
    index=False
)

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 70)
print("STEP 11F — PER-CLASS ROBUSTNESS ANALYSIS")
print("=" * 70)

print()
print("Input:")
print(INPUT_PATH_11F)

print()
print("Output:")
print(OUTPUT_PATH_11F)

print()
print("=" * 70)
print("AUROC BY CLASS AND SNR")
print("=" * 70)
print()

print(
    auroc_table_11F.to_string(
        float_format=lambda x: f"{x:.6f}"
    )
)

print()
print("=" * 70)
print("AUPRC BY CLASS AND SNR")
print("=" * 70)
print()

print(
    auprc_table_11F.to_string(
        float_format=lambda x: f"{x:.6f}"
    )
)

print()
print("=" * 70)
print("F1 BY CLASS AND SNR")
print("=" * 70)
print()

print(
    f1_table_11F.to_string(
        float_format=lambda x: f"{x:.6f}"
    )
)

print()
print("=" * 70)
print("0 dB — PER-CLASS DEGRADATION")
print("=" * 70)
print()

print(
    zero_db_11F.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print()
print("=" * 70)
print("ROBUSTNESS FINDINGS")
print("=" * 70)

print()
print(
    "Largest AUROC degradation at 0 dB:",
    largest_auroc_drop_class_11F,
    f"({largest_auroc_drop_11F:.6f})"
)

print(
    "Largest AUPRC degradation at 0 dB:",
    largest_auprc_drop_class_11F,
    f"({largest_auprc_drop_11F:.6f})"
)

print()
print(
    "All five diagnostic classes retained "
    "AUROC > 0.89 at 0 dB:",
    bool(
        (zero_db_11F["AUROC"] > 0.89).all()
    )
)

print()
print("=" * 70)
print("STEP 11F STATUS: PASS")
print("=" * 70)

STEP 11F — PER-CLASS ROBUSTNESS ANALYSIS

Input:
/content/drive/MyDrive/PTB-XL Research Project/results/noise_robustness/step11E_noise_robustness_results.csv

Output:
/content/drive/MyDrive/PTB-XL Research Project/results/noise_robustness/step11F_per_class_robustness.csv

AUROC BY CLASS AND SNR

condition    clean  snr_30db  snr_20db  snr_10db  snr_0db
target                                                   
MI        0.934836  0.934726  0.934447  0.933889 0.919003
STTC      0.933429  0.933391  0.933391  0.931979 0.920288
CD        0.924570  0.924663  0.924368  0.923509 0.912122
HYP       0.909130  0.909117  0.909209  0.908490 0.898307
NORM      0.948333  0.948337  0.948219  0.947653 0.937512

AUPRC BY CLASS AND SNR

condition    clean  snr_30db  snr_20db  snr_10db  snr_0db
target                                                   
MI        0.850112  0.850098  0.848988  0.848928 0.819559
STTC      0.823583  0.823516  0.824131  0.821692 0.798478
CD        0.843310  0.843321  0.843372  

In [15]:
# ============================================================
# STEP 11G — ROBUSTNESS CURVES & QUANTITATIVE DEGRADATION
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_PATH_11G = (
    "/content/drive/MyDrive/PTB-XL Research Project"
)

NOISE_RESULTS_PATH_11G = os.path.join(
    PROJECT_PATH_11G,
    "results",
    "noise_robustness"
)

INPUT_PATH_11G = os.path.join(
    NOISE_RESULTS_PATH_11G,
    "step11E_noise_robustness_results.csv"
)

OUTPUT_CURVE_PATH_11G = os.path.join(
    NOISE_RESULTS_PATH_11G,
    "step11G_robustness_curves.csv"
)

OUTPUT_DEGRADATION_PATH_11G = os.path.join(
    NOISE_RESULTS_PATH_11G,
    "step11G_degradation_analysis.csv"
)

OUTPUT_SUMMARY_PATH_11G = os.path.join(
    NOISE_RESULTS_PATH_11G,
    "step11G_robustness_summary.csv"
)

# ------------------------------------------------------------
# LOAD FROZEN 11E RESULTS
# ------------------------------------------------------------

assert os.path.exists(INPUT_PATH_11G), (
    f"11E results not found:\n{INPUT_PATH_11G}"
)

df_11G = pd.read_csv(
    INPUT_PATH_11G
)

EXPECTED_CLASSES_11G = [
    "MI",
    "STTC",
    "CD",
    "HYP",
    "NORM"
]

EXPECTED_CONDITIONS_11G = [
    "clean",
    "snr_30db",
    "snr_20db",
    "snr_10db",
    "snr_0db"
]

assert len(df_11G) == 30

assert set(
    df_11G["target"].unique()
) == set(
    EXPECTED_CLASSES_11G + ["MACRO"]
)

# ------------------------------------------------------------
# NUMERIC SNR REPRESENTATION
# ------------------------------------------------------------
# Clean is represented separately as the highest-quality
# reference condition. For ordered robustness curves:
#
# clean -> 30 -> 20 -> 10 -> 0 dB
#
# We use a numeric plotting/evaluation value of 40 dB for
# "clean" ONLY as an analysis label, not as a measured SNR.
# The original condition remains preserved.

snr_map_11G = {
    "clean": 40.0,
    "snr_30db": 30.0,
    "snr_20db": 20.0,
    "snr_10db": 10.0,
    "snr_0db": 0.0
}

condition_order_11G = [
    "clean",
    "snr_30db",
    "snr_20db",
    "snr_10db",
    "snr_0db"
]

# ------------------------------------------------------------
# MACRO ROBUSTNESS CURVES
# ------------------------------------------------------------

macro_df_11G = (
    df_11G[
        df_11G["target"] == "MACRO"
    ]
    .copy()
)

macro_df_11G["analysis_snr_db"] = (
    macro_df_11G["condition"]
    .map(snr_map_11G)
)

macro_df_11G["condition_order"] = (
    macro_df_11G["condition"]
    .map(
        {
            condition: i
            for i, condition
            in enumerate(condition_order_11G)
        }
    )
)

macro_df_11G = (
    macro_df_11G
    .sort_values("condition_order")
)

# ------------------------------------------------------------
# PER-CLASS CURVES
# ------------------------------------------------------------

class_df_11G = (
    df_11G[
        df_11G["target"].isin(
            EXPECTED_CLASSES_11G
        )
    ]
    .copy()
)

class_df_11G["analysis_snr_db"] = (
    class_df_11G["condition"]
    .map(snr_map_11G)
)

class_df_11G["condition_order"] = (
    class_df_11G["condition"]
    .map(
        {
            condition: i
            for i, condition
            in enumerate(condition_order_11G)
        }
    )
)

class_df_11G = (
    class_df_11G
    .sort_values(
        [
            "target",
            "condition_order"
        ]
    )
)

# ------------------------------------------------------------
# CREATE ROBUSTNESS CURVE ARTIFACT
# ------------------------------------------------------------

curve_columns_11G = [
    "condition",
    "analysis_snr_db",
    "target",
    "AUROC",
    "AUPRC",
    "F1",
    "sensitivity",
    "specificity"
]

curve_df_11G = (
    pd.concat(
        [
            macro_df_11G,
            class_df_11G
        ],
        ignore_index=True
    )
    [curve_columns_11G]
)

curve_df_11G.to_csv(
    OUTPUT_CURVE_PATH_11G,
    index=False
)

# ------------------------------------------------------------
# DEGRADATION ANALYSIS
# ------------------------------------------------------------

degradation_rows_11G = []

for target in (
    EXPECTED_CLASSES_11G + ["MACRO"]
):

    target_df = (
        df_11G[
            df_11G["target"] == target
        ]
        .copy()
    )

    clean_row = target_df[
        target_df["condition"] == "clean"
    ].iloc[0]

    clean_auroc = float(
        clean_row["AUROC"]
    )

    clean_auprc = float(
        clean_row["AUPRC"]
    )

    clean_f1 = float(
        clean_row["F1"]
    )

    for condition in condition_order_11G:

        row = target_df[
            target_df["condition"] == condition
        ].iloc[0]

        current_auroc = float(
            row["AUROC"]
        )

        current_auprc = float(
            row["AUPRC"]
        )

        current_f1 = float(
            row["F1"]
        )

        degradation_rows_11G.append({

            "target": target,

            "condition": condition,

            "analysis_snr_db":
                snr_map_11G[condition],

            "AUROC":
                current_auroc,

            "AUPRC":
                current_auprc,

            "F1":
                current_f1,

            "AUROC_drop_from_clean":
                clean_auroc - current_auroc,

            "AUPRC_drop_from_clean":
                clean_auprc - current_auprc,

            "F1_drop_from_clean":
                clean_f1 - current_f1,

            "AUROC_relative_drop_percent":
                (
                    (clean_auroc - current_auroc)
                    / clean_auroc
                    * 100.0
                ),

            "AUPRC_relative_drop_percent":
                (
                    (clean_auprc - current_auprc)
                    / clean_auprc
                    * 100.0
                ),

            "F1_relative_drop_percent":
                (
                    (clean_f1 - current_f1)
                    / clean_f1
                    * 100.0
                )
        })

degradation_df_11G = pd.DataFrame(
    degradation_rows_11G
)

degradation_df_11G.to_csv(
    OUTPUT_DEGRADATION_PATH_11G,
    index=False
)

# ------------------------------------------------------------
# SNR-SPECIFIC MACRO DEGRADATION
# ------------------------------------------------------------

macro_degradation_11G = (
    degradation_df_11G[
        degradation_df_11G["target"] == "MACRO"
    ]
    .copy()
)

# ------------------------------------------------------------
# DETERMINE FIRST CONDITION WITH MEANINGFUL DEGRADATION
# ------------------------------------------------------------
# Thresholds are descriptive analysis thresholds:
#
# AUROC absolute drop >= 0.01
# AUPRC absolute drop >= 0.02
#
# These are NOT statistical significance thresholds.

auroc_threshold_11G = 0.01
auprc_threshold_11G = 0.02

meaningful_auroc_11G = (
    macro_degradation_11G[
        macro_degradation_11G[
            "AUROC_drop_from_clean"
        ] >= auroc_threshold_11G
    ]
)

meaningful_auprc_11G = (
    macro_degradation_11G[
        macro_degradation_11G[
            "AUPRC_drop_from_clean"
        ] >= auprc_threshold_11G
    ]
)

if len(meaningful_auroc_11G) > 0:

    first_auroc_condition_11G = (
        meaningful_auroc_11G.iloc[0][
            "condition"
        ]
    )

    first_auroc_snr_11G = float(
        meaningful_auroc_11G.iloc[0][
            "analysis_snr_db"
        ]
    )

else:

    first_auroc_condition_11G = (
        "none within tested conditions"
    )

    first_auroc_snr_11G = np.nan


if len(meaningful_auprc_11G) > 0:

    first_auprc_condition_11G = (
        meaningful_auprc_11G.iloc[0][
            "condition"
        ]
    )

    first_auprc_snr_11G = float(
        meaningful_auprc_11G.iloc[0][
            "analysis_snr_db"
        ]
    )

else:

    first_auprc_condition_11G = (
        "none within tested conditions"
    )

    first_auprc_snr_11G = np.nan

# ------------------------------------------------------------
# CALCULATE 0 dB DEGRADATION RANKING
# ------------------------------------------------------------

zero_db_11G = (
    degradation_df_11G[
        (
            degradation_df_11G["condition"]
            == "snr_0db"
        )
    ]
    .copy()
)

zero_db_11G = (
    zero_db_11G
    .sort_values(
        "AUROC_drop_from_clean",
        ascending=False
    )
)

# ------------------------------------------------------------
# DESCRIPTIVE LINEAR SLOPE
# ------------------------------------------------------------
# This is a descriptive slope across the tested SNR
# conditions only. It is NOT intended as a mechanistic
# or population-level model.

slope_rows_11G = []

for target in (
    EXPECTED_CLASSES_11G + ["MACRO"]
):

    target_df = (
        degradation_df_11G[
            degradation_df_11G["target"] == target
        ]
        .copy()
    )

    x = target_df[
        "analysis_snr_db"
    ].to_numpy(
        dtype=float
    )

    y_auroc = target_df[
        "AUROC"
    ].to_numpy(
        dtype=float
    )

    y_auprc = target_df[
        "AUPRC"
    ].to_numpy(
        dtype=float
    )

    y_f1 = target_df[
        "F1"
    ].to_numpy(
        dtype=float
    )

    auroc_slope = np.polyfit(
        x,
        y_auroc,
        1
    )[0]

    auprc_slope = np.polyfit(
        x,
        y_auprc,
        1
    )[0]

    f1_slope = np.polyfit(
        x,
        y_f1,
        1
    )[0]

    slope_rows_11G.append({

        "target": target,

        "AUROC_slope_per_dB":
            auroc_slope,

        "AUPRC_slope_per_dB":
            auprc_slope,

        "F1_slope_per_dB":
            f1_slope,

        "AUROC_change_40_to_0db":
            y_auroc[-1] - y_auroc[0],

        "AUPRC_change_40_to_0db":
            y_auprc[-1] - y_auprc[0],

        "F1_change_40_to_0db":
            y_f1[-1] - y_f1[0]
    })

slope_df_11G = pd.DataFrame(
    slope_rows_11G
)

# ------------------------------------------------------------
# COMBINE SUMMARY
# ------------------------------------------------------------

summary_rows_11G = []

for target in (
    EXPECTED_CLASSES_11G + ["MACRO"]
):

    clean = degradation_df_11G[
        (
            degradation_df_11G["target"]
            == target
        )
        &
        (
            degradation_df_11G["condition"]
            == "clean"
        )
    ].iloc[0]

    zero = degradation_df_11G[
        (
            degradation_df_11G["target"]
            == target
        )
        &
        (
            degradation_df_11G["condition"]
            == "snr_0db"
        )
    ].iloc[0]

    slope = slope_df_11G[
        slope_df_11G["target"] == target
    ].iloc[0]

    summary_rows_11G.append({

        "target": target,

        "clean_AUROC":
            clean["AUROC"],

        "zero_db_AUROC":
            zero["AUROC"],

        "AUROC_drop":
            zero["AUROC_drop_from_clean"],

        "AUROC_relative_drop_percent":
            zero["AUROC_relative_drop_percent"],

        "clean_AUPRC":
            clean["AUPRC"],

        "zero_db_AUPRC":
            zero["AUPRC"],

        "AUPRC_drop":
            zero["AUPRC_drop_from_clean"],

        "AUPRC_relative_drop_percent":
            zero["AUPRC_relative_drop_percent"],

        "clean_F1":
            clean["F1"],

        "zero_db_F1":
            zero["F1"],

        "F1_drop":
            zero["F1_drop_from_clean"],

        "F1_relative_drop_percent":
            zero["F1_relative_drop_percent"],

        "AUROC_slope_per_dB":
            slope["AUROC_slope_per_dB"],

        "AUPRC_slope_per_dB":
            slope["AUPRC_slope_per_dB"],

        "F1_slope_per_dB":
            slope["F1_slope_per_dB"]
    })

summary_df_11G = pd.DataFrame(
    summary_rows_11G
)

summary_df_11G.to_csv(
    OUTPUT_SUMMARY_PATH_11G,
    index=False
)

# ------------------------------------------------------------
# INTEGRITY CHECKS
# ------------------------------------------------------------

assert len(curve_df_11G) == 30

assert len(
    degradation_df_11G
) == 30

assert len(
    summary_df_11G
) == 6

assert np.isfinite(
    degradation_df_11G[
        [
            "AUROC",
            "AUPRC",
            "F1",
            "AUROC_drop_from_clean",
            "AUPRC_drop_from_clean",
            "F1_drop_from_clean"
        ]
    ]
    .to_numpy()
).all()

# Verify the known macro values from 11E

macro_clean_11G = summary_df_11G[
    summary_df_11G["target"] == "MACRO"
].iloc[0]

assert abs(
    macro_clean_11G["clean_AUROC"]
    - 0.930060
) < 1e-6

assert abs(
    macro_clean_11G["zero_db_AUROC"]
    - 0.917446
) < 1e-6

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("=" * 70)
print("STEP 11G — ROBUSTNESS CURVES & QUANTITATIVE DEGRADATION")
print("=" * 70)

print()
print(
    "Important: 'clean' is displayed as 40 dB only as an "
    "ordering/reference label."
)
print(
    "It is NOT being claimed to have a measured 40 dB SNR."
)

print()
print("=" * 70)
print("MACRO ROBUSTNESS CURVE DATA")
print("=" * 70)
print()

print(
    macro_df_11G[
        [
            "condition",
            "analysis_snr_db",
            "AUROC",
            "AUPRC",
            "F1",
            "sensitivity",
            "specificity"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print()
print("=" * 70)
print("0 dB CLASS RANKING BY AUROC DEGRADATION")
print("=" * 70)
print()

print(
    zero_db_11G[
        [
            "target",
            "AUROC",
            "AUROC_drop_from_clean",
            "AUROC_relative_drop_percent",
            "AUPRC_drop_from_clean",
            "F1_drop_from_clean"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print()
print("=" * 70)
print("DESCRIPTIVE SLOPES")
print("=" * 70)
print()

print(
    slope_df_11G.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

print()
print("=" * 70)
print("MACRO DEGRADATION THRESHOLDS")
print("=" * 70)

print()
print(
    "AUROC threshold:",
    f"{auroc_threshold_11G:.3f}"
)

print(
    "AUPRC threshold:",
    f"{auprc_threshold_11G:.3f}"
)

print()
print(
    "First tested condition with macro-AUROC "
    "drop >= 0.01:",
    first_auroc_condition_11G
)

print(
    "First tested SNR label:",
    first_auroc_snr_11G
)

print(
    "First tested condition with macro-AUPRC "
    "drop >= 0.02:",
    first_auprc_condition_11G
)

print(
    "First tested SNR label:",
    first_auprc_snr_11G
)

print()
print("=" * 70)
print("SAVED ARTIFACTS")
print("=" * 70)

print()
print(OUTPUT_CURVE_PATH_11G)
print(OUTPUT_DEGRADATION_PATH_11G)
print(OUTPUT_SUMMARY_PATH_11G)

print()
print("=" * 70)
print("STEP 11G STATUS: PASS")
print("=" * 70)

STEP 11G — ROBUSTNESS CURVES & QUANTITATIVE DEGRADATION

Important: 'clean' is displayed as 40 dB only as an ordering/reference label.
It is NOT being claimed to have a measured 40 dB SNR.

MACRO ROBUSTNESS CURVE DATA

condition  analysis_snr_db    AUROC    AUPRC       F1  sensitivity  specificity
    clean        40.000000 0.930060 0.824810 0.718363     0.852513     0.851310
 snr_30db        30.000000 0.930047 0.824706 0.718211     0.853261     0.850664
 snr_20db        20.000000 0.929927 0.824631 0.717784     0.852861     0.850339
 snr_10db        10.000000 0.929104 0.824244 0.716273     0.853539     0.849235
  snr_0db         0.000000 0.917446 0.800443 0.677490     0.814736     0.829534

0 dB CLASS RANKING BY AUROC DEGRADATION

target    AUROC  AUROC_drop_from_clean  AUROC_relative_drop_percent  AUPRC_drop_from_clean  F1_drop_from_clean
    MI 0.919003               0.015833                     1.693663               0.030553            0.013555
  STTC 0.920288               0.01314

In [23]:
# ============================================================
# STEP 11H-1 — SETUP & FROZEN MODEL VERIFICATION
# CORRECTED: RESIDUAL BLOCK KERNEL SIZE = 7
# ============================================================

import os
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wfdb

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_PATH_11H = (
    "/content/drive/MyDrive/PTB-XL Research Project"
)

DATA_PATH_11H = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
)

RESULTS_PATH_11H = (
    "/content/drive/MyDrive/PTB-XL Research Project/results"
)

DEEP_LEARNING_PATH_11H = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "results/deep_learning"
)

NOISE_RESULTS_PATH_11H = (
    "/content/drive/MyDrive/PTB-XL Research Project/"
    "results/noise_robustness"
)

CHECKPOINT_11H = os.path.join(
    DEEP_LEARNING_PATH_11H,
    "ecg_resnet1d_best_validation.pt"
)

SPLIT_PATH_11H = os.path.join(
    RESULTS_PATH_11H,
    "ptbxl_patient_independent_split.csv"
)

TARGETS_PATH_11H = os.path.join(
    RESULTS_PATH_11H,
    "ptbxl_diagnostic_targets.csv"
)

DATABASE_PATH_11H = os.path.join(
    DATA_PATH_11H,
    "ptbxl_database.csv"
)

NOISE_11E_PATH = os.path.join(
    NOISE_RESULTS_PATH_11H,
    "step11E_noise_robustness_results.csv"
)

RESULTS_11H = os.path.join(
    NOISE_RESULTS_PATH_11H,
    "step11H_preprocessing_ablation_results.csv"
)

DETAILS_11H = os.path.join(
    NOISE_RESULTS_PATH_11H,
    "step11H_preprocessing_ablation_details.csv"
)

TARGET_COLUMNS_11H = [
    "MI",
    "STTC",
    "CD",
    "HYP",
    "NORM"
]

FS_11H = 500
N_LEADS_11H = 12
N_SAMPLES_11H = 5000

# ------------------------------------------------------------
# 2. VERIFY PATHS
# ------------------------------------------------------------

required_paths_11H = {
    "Dataset": DATA_PATH_11H,
    "PTB-XL database": DATABASE_PATH_11H,
    "Patient-independent split": SPLIT_PATH_11H,
    "Diagnostic targets": TARGETS_PATH_11H,
    "Frozen checkpoint": CHECKPOINT_11H,
    "11E noise results": NOISE_11E_PATH,
}

print("=" * 70)
print("STEP 11H-1 — SETUP & FROZEN MODEL VERIFICATION")
print("=" * 70)

print("\nPATH VERIFICATION")
print("-" * 70)

all_paths_ok_11H = True

for name, path in required_paths_11H.items():
    exists = os.path.exists(path)
    print(f"{name:<30}: {'PASS' if exists else 'FAIL'}")

    if not exists:
        print(f"  {path}")
        all_paths_ok_11H = False

if not all_paths_ok_11H:
    raise FileNotFoundError(
        "One or more required project paths are missing."
    )

print("\nAll required paths: PASS")

# ------------------------------------------------------------
# 3. EXACT NOTEBOOK 10 RESNET ARCHITECTURE
# ------------------------------------------------------------

class ResidualBlock1D(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=7,
            stride=stride,
            padding=3,
            bias=False
        )

        self.bn1 = nn.BatchNorm1d(out_channels)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=7,
            stride=1,
            padding=3,
            bias=False
        )

        self.bn2 = nn.BatchNorm1d(out_channels)

        if stride != 1 or in_channels != out_channels:

            self.shortcut = nn.Sequential(
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm1d(out_channels)
            )

        else:
            self.shortcut = nn.Identity()

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):

        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)

        return out


class ECGResNet1D(nn.Module):

    def __init__(self, num_classes=5):

        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(
                12,
                64,
                kernel_size=15,
                stride=2,
                padding=7,
                bias=False
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(
                kernel_size=3,
                stride=2,
                padding=1
            )
        )

        self.layer1 = nn.Sequential(
            ResidualBlock1D(64, 64, stride=1),
            ResidualBlock1D(64, 64, stride=1)
        )

        self.layer2 = nn.Sequential(
            ResidualBlock1D(64, 128, stride=2),
            ResidualBlock1D(128, 128, stride=1)
        )

        self.layer3 = nn.Sequential(
            ResidualBlock1D(128, 256, stride=2),
            ResidualBlock1D(256, 256, stride=1)
        )

        self.layer4 = nn.Sequential(
            ResidualBlock1D(256, 512, stride=2),
            ResidualBlock1D(512, 512, stride=1)
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Linear(
            512,
            num_classes
        )

    def forward(self, x):

        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.global_pool(x)

        x = torch.flatten(x, 1)

        x = self.classifier(x)

        return x


# ------------------------------------------------------------
# 4. LOAD FROZEN CHECKPOINT
# ------------------------------------------------------------

device_11H = torch.device("cpu")

model_11H = ECGResNet1D(
    num_classes=len(TARGET_COLUMNS_11H)
)

checkpoint_11H = torch.load(
    CHECKPOINT_11H,
    map_location=device_11H,
    weights_only=False
)

model_11H.load_state_dict(
    checkpoint_11H["model_state_dict"],
    strict=True
)

model_11H.to(device_11H)
model_11H.eval()

for parameter in model_11H.parameters():
    parameter.requires_grad = False

# ------------------------------------------------------------
# 5. VERIFY MODEL
# ------------------------------------------------------------

parameter_count_11H = sum(
    p.numel()
    for p in model_11H.parameters()
)

checkpoint_epoch_11H = checkpoint_11H["epoch"]

best_val_auroc_11H = (
    checkpoint_11H["best_validation_macro_auroc"]
)

print("\nFROZEN MODEL VERIFICATION")
print("-" * 70)

print(
    f"Architecture:              ECGResNet1D"
)

print(
    f"Parameters:                {parameter_count_11H:,}"
)

print(
    f"Checkpoint epoch:          {checkpoint_epoch_11H}"
)

print(
    f"Best validation AUROC:     {best_val_auroc_11H:.6f}"
)

assert parameter_count_11H == 8_739_973
assert checkpoint_epoch_11H == 17
assert abs(best_val_auroc_11H - 0.934005) < 1e-6

print("Strict checkpoint loading: PASS")
print("Parameters frozen:         PASS")

# ------------------------------------------------------------
# 6. LOAD DATA FILES
# ------------------------------------------------------------

split_11H = pd.read_csv(
    SPLIT_PATH_11H
)

targets_11H = pd.read_csv(
    TARGETS_PATH_11H
)

database_11H = pd.read_csv(
    DATABASE_PATH_11H
)

print("\nDATA FILES")
print("-" * 70)

print(
    f"Split rows:                {len(split_11H):,}"
)

print(
    f"Target rows:               {len(targets_11H):,}"
)

print(
    f"Database rows:             {len(database_11H):,}"
)

# ------------------------------------------------------------
# 7. IDENTIFY LOCKED TEST SET
# ------------------------------------------------------------

test_ids_11H = split_11H.loc[
    split_11H["split"].astype(str).str.lower() == "test",
    "ecg_id"
].astype(int).tolist()

print("\nLOCKED TEST SET")
print("-" * 70)

print(
    f"Test ECGs:                 {len(test_ids_11H):,}"
)

assert len(test_ids_11H) == 2198

# ------------------------------------------------------------
# 8. MERGE TEST METADATA + TARGETS
# ------------------------------------------------------------

test_targets_11H = targets_11H[
    targets_11H["ecg_id"].isin(test_ids_11H)
].copy()

test_database_11H = database_11H[
    database_11H["ecg_id"].isin(test_ids_11H)
].copy()

assert len(test_targets_11H) == 2198
assert len(test_database_11H) == 2198

test_metadata_11H = test_database_11H.merge(
    test_targets_11H,
    on="ecg_id",
    how="inner",
    validate="one_to_one"
)

assert len(test_metadata_11H) == 2198

# ------------------------------------------------------------
# 9. VERIFY CANONICAL HR FILE NAMES
# ------------------------------------------------------------

assert "filename_hr" in test_metadata_11H.columns

missing_hr_11H = (
    test_metadata_11H["filename_hr"]
    .isna()
    .sum()
)

assert missing_hr_11H == 0

print("Canonical filename_hr:     PASS")
print("Target mapping:            PASS")
print("Test-set integrity:        PASS")

# ------------------------------------------------------------
# 10. VERIFY TARGET COLUMNS
# ------------------------------------------------------------

missing_targets_11H = [
    c for c in TARGET_COLUMNS_11H
    if c not in test_metadata_11H.columns
]

assert len(missing_targets_11H) == 0

print("Target columns:             PASS")

# ------------------------------------------------------------
# 11. CHECKPOINT SHA256
# ------------------------------------------------------------

sha256_11H = hashlib.sha256()

with open(CHECKPOINT_11H, "rb") as f:

    for chunk in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):
        sha256_11H.update(chunk)

checkpoint_hash_11H = sha256_11H.hexdigest()

print("\nCHECKPOINT INTEGRITY")
print("-" * 70)

print(
    f"SHA256: {checkpoint_hash_11H}"
)

assert checkpoint_hash_11H == (
    "d0937a7ffb2b41662fcce1fc3d6358540cbac3f185989e23f916f0084d7492c4"
)

print("Checkpoint SHA256:         PASS")

# ------------------------------------------------------------
# 12. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 11H-1 STATUS: PASS")
print("=" * 70)

print("Frozen model verified.")
print("Locked test set verified.")
print("Canonical filename_hr verified.")
print("No model parameters modified.")
print("No experiment has been run yet.")

STEP 11H-1 — SETUP & FROZEN MODEL VERIFICATION

PATH VERIFICATION
----------------------------------------------------------------------
Dataset                       : PASS
PTB-XL database               : PASS
Patient-independent split     : PASS
Diagnostic targets            : PASS
Frozen checkpoint             : PASS
11E noise results             : PASS

All required paths: PASS

FROZEN MODEL VERIFICATION
----------------------------------------------------------------------
Architecture:              ECGResNet1D
Parameters:                8,739,973
Checkpoint epoch:          17
Best validation AUROC:     0.934005
Strict checkpoint loading: PASS
Parameters frozen:         PASS

DATA FILES
----------------------------------------------------------------------
Split rows:                21,799
Target rows:               21,799
Database rows:             21,799

LOCKED TEST SET
----------------------------------------------------------------------
Test ECGs:                 2,198
Canon

In [24]:
# ============================================================
# STEP 11H-2 — PREPROCESSING PREPARATION
# ============================================================

from scipy import signal

# ------------------------------------------------------------
# 1. FILTER CONFIGURATION
# ------------------------------------------------------------

LOWCUT_CANDIDATES_11H = [
    0.5,
    0.6,
    0.7,
    0.8
]

HIGHCUT_11H = 40.0
FILTER_ORDER_11H = 4

MORPHOLOGY_THRESHOLD_11H = 0.999
MIN_BASELINE_GAIN_PP_11H = 0.50

LEAD_NAMES_11H = [
    "I", "II", "III", "AVR", "AVL", "AVF",
    "V1", "V2", "V3", "V4", "V5", "V6"
]

print("=" * 70)
print("STEP 11H-2 — PREPROCESSING PREPARATION")
print("=" * 70)

# ------------------------------------------------------------
# 2. PRECOMPUTE BAND-PASS FILTER COEFFICIENTS
# ------------------------------------------------------------

bandpass_coefficients_11H = {}

nyquist_11H = 0.5 * FS_11H

for cutoff in LOWCUT_CANDIDATES_11H:

    low = cutoff / nyquist_11H
    high = HIGHCUT_11H / nyquist_11H

    b, a = signal.butter(
        FILTER_ORDER_11H,
        [low, high],
        btype="bandpass"
    )

    bandpass_coefficients_11H[cutoff] = (b, a)

print("\nBAND-PASS FILTERS")
print("-" * 70)

for cutoff in LOWCUT_CANDIDATES_11H:
    print(
        f"{cutoff:.1f} Hz → 40 Hz: PRECOMPUTED"
    )

# ------------------------------------------------------------
# 3. PRECOMPUTE BASELINE-ESTIMATION FILTER
# ------------------------------------------------------------

baseline_sos_11H = signal.butter(
    3,
    0.3 / nyquist_11H,
    btype="lowpass",
    output="sos"
)

print(
    "\n0.3 Hz baseline filter: PRECOMPUTED"
)

# ------------------------------------------------------------
# 4. EXACT BAND-PASS FUNCTION
# ------------------------------------------------------------

def bandpass_filter_11H(
    ecg_signal,
    cutoff,
    coefficients=bandpass_coefficients_11H
):

    b, a = coefficients[cutoff]

    return signal.filtfilt(
        b,
        a,
        ecg_signal,
        axis=0
    )


# ------------------------------------------------------------
# 5. MORPHOLOGY SIMILARITY
# ------------------------------------------------------------

def morphology_similarity_11H(
    reference,
    candidate
):

    reference = np.asarray(reference)
    candidate = np.asarray(candidate)

    reference_centered = (
        reference - np.mean(reference)
    )

    candidate_centered = (
        candidate - np.mean(candidate)
    )

    denominator = (
        np.sqrt(
            np.sum(reference_centered ** 2)
        )
        *
        np.sqrt(
            np.sum(candidate_centered ** 2)
        )
    )

    if denominator == 0:
        return 1.0 if np.array_equal(
            reference,
            candidate
        ) else 0.0

    return (
        np.sum(
            reference_centered * candidate_centered
        )
        / denominator
    )


# ------------------------------------------------------------
# 6. BASELINE SUPPRESSION GAIN
# ------------------------------------------------------------

def baseline_suppression_gain_pp_11H(
    original_signal,
    filtered_signal
):

    original_baseline = signal.sosfiltfilt(
        baseline_sos_11H,
        original_signal,
        axis=0
    )

    filtered_baseline = signal.sosfiltfilt(
        baseline_sos_11H,
        filtered_signal,
        axis=0
    )

    original_rms = np.sqrt(
        np.mean(
            original_baseline ** 2,
            axis=0
        )
    )

    filtered_rms = np.sqrt(
        np.mean(
            filtered_baseline ** 2,
            axis=0
        )
    )

    epsilon = 1e-12

    suppression_original = (
        100.0
        * original_rms
        / (
            np.sqrt(
                np.mean(
                    original_signal ** 2,
                    axis=0
                )
            )
            + epsilon
        )
    )

    suppression_filtered = (
        100.0
        * filtered_rms
        / (
            np.sqrt(
                np.mean(
                    filtered_signal ** 2,
                    axis=0
                )
            )
            + epsilon
        )
    )

    gain_pp = (
        suppression_original
        - suppression_filtered
    )

    return gain_pp


# ------------------------------------------------------------
# 7. EXACT LOCKED ADAPTIVE CUTOFF SELECTION
# ------------------------------------------------------------

def select_adaptive_cutoff_11H(
    ecg_signal
):

    # Fixed 0.5 Hz reference
    reference_05 = bandpass_filter_11H(
        ecg_signal,
        0.5
    )

    # Baseline of original signal
    original_baseline = signal.sosfiltfilt(
        baseline_sos_11H,
        ecg_signal,
        axis=0
    )

    original_rms = np.sqrt(
        np.mean(
            original_baseline ** 2,
            axis=0
        )
    )

    original_signal_rms = np.sqrt(
        np.mean(
            ecg_signal ** 2,
            axis=0
        )
    )

    epsilon = 1e-12

    original_baseline_ratio = (
        original_rms
        /
        (original_signal_rms + epsilon)
    )

    selected_cutoffs = np.full(
        N_LEADS_11H,
        0.5,
        dtype=np.float64
    )

    for cutoff in LOWCUT_CANDIDATES_11H:

        candidate = bandpass_filter_11H(
            ecg_signal,
            cutoff
        )

        # ----------------------------------------------------
        # Morphology similarity to locked 0.5 Hz reference
        # ----------------------------------------------------

        morphology_ok = np.zeros(
            N_LEADS_11H,
            dtype=bool
        )

        for lead_idx in range(N_LEADS_11H):

            similarity = morphology_similarity_11H(
                reference_05[:, lead_idx],
                candidate[:, lead_idx]
            )

            morphology_ok[lead_idx] = (
                similarity
                >= MORPHOLOGY_THRESHOLD_11H
            )

        # ----------------------------------------------------
        # Baseline suppression gain
        # ----------------------------------------------------

        candidate_baseline = signal.sosfiltfilt(
            baseline_sos_11H,
            candidate,
            axis=0
        )

        candidate_rms = np.sqrt(
            np.mean(
                candidate_baseline ** 2,
                axis=0
            )
        )

        candidate_signal_rms = np.sqrt(
            np.mean(
                candidate ** 2,
                axis=0
            )
        )

        candidate_baseline_ratio = (
            candidate_rms
            /
            (candidate_signal_rms + epsilon)
        )

        gain_pp = 100.0 * (
            original_baseline_ratio
            -
            candidate_baseline_ratio
        )

        baseline_ok = (
            gain_pp
            >= MIN_BASELINE_GAIN_PP_11H
        )

        valid = (
            morphology_ok
            &
            baseline_ok
        )

        # Highest valid candidate wins
        selected_cutoffs[valid] = cutoff

    return selected_cutoffs


# ------------------------------------------------------------
# 8. FIXED FILTER
# ------------------------------------------------------------

def apply_fixed_filter_11H(
    ecg_signal
):

    return bandpass_filter_11H(
        ecg_signal,
        0.5
    )


# ------------------------------------------------------------
# 9. ADAPTIVE FILTER
# ------------------------------------------------------------

def apply_adaptive_filter_11H(
    ecg_signal,
    cutoff_per_lead
):

    filtered = np.empty_like(
        ecg_signal,
        dtype=np.float32
    )

    for cutoff in LOWCUT_CANDIDATES_11H:

        lead_indices = np.where(
            cutoff_per_lead == cutoff
        )[0]

        if len(lead_indices) == 0:
            continue

        candidate = bandpass_filter_11H(
            ecg_signal,
            cutoff
        )

        filtered[:, lead_indices] = (
            candidate[:, lead_indices]
        )

    return filtered


# ------------------------------------------------------------
# 10. TEST ON ONE LOCKED ECG
# ------------------------------------------------------------

test_row_11H = test_metadata_11H.iloc[0]

ecg_id_check_11H = int(
    test_row_11H["ecg_id"]
)

record_path_check_11H = os.path.join(
    DATA_PATH_11H,
    str(test_row_11H["filename_hr"])
)

signal_check_11H, signal_fields_check_11H = (
    wfdb.rdsamp(record_path_check_11H)
)

signal_check_11H = np.asarray(
    signal_check_11H,
    dtype=np.float32
)

assert signal_check_11H.shape == (
    N_SAMPLES_11H,
    N_LEADS_11H
)

cutoffs_check_11H = select_adaptive_cutoff_11H(
    signal_check_11H
)

fixed_check_11H = apply_fixed_filter_11H(
    signal_check_11H
)

adaptive_check_11H = apply_adaptive_filter_11H(
    signal_check_11H,
    cutoffs_check_11H
)

assert fixed_check_11H.shape == (
    N_SAMPLES_11H,
    N_LEADS_11H
)

assert adaptive_check_11H.shape == (
    N_SAMPLES_11H,
    N_LEADS_11H
)

assert np.all(
    np.isfinite(fixed_check_11H)
)

assert np.all(
    np.isfinite(adaptive_check_11H)
)

# ------------------------------------------------------------
# 11. FINAL STATUS
# ------------------------------------------------------------

print("\nSINGLE-ECG FILTER TEST")
print("-" * 70)

print(
    f"ECG ID:                    {ecg_id_check_11H}"
)

print(
    f"Input shape:               {signal_check_11H.shape}"
)

print(
    "Fixed 0.5 Hz filter:       PASS"
)

print(
    "Adaptive filter:           PASS"
)

print(
    "Finite outputs:            PASS"
)

print(
    "\nAdaptive cutoffs for test ECG:"
)

for lead_name, cutoff in zip(
    LEAD_NAMES_11H,
    cutoffs_check_11H
):
    print(
        f"  {lead_name:>3}: {cutoff:.1f} Hz"
    )

print("\n" + "=" * 70)
print("STEP 11H-2 STATUS: PASS")
print("=" * 70)

print(
    "Filter coefficients precomputed."
)

print(
    "Locked adaptive selection verified."
)

print(
    "No ResNet evaluation performed."
)

print(
    "No test-set predictions generated."
)

STEP 11H-2 — PREPROCESSING PREPARATION

BAND-PASS FILTERS
----------------------------------------------------------------------
0.5 Hz → 40 Hz: PRECOMPUTED
0.6 Hz → 40 Hz: PRECOMPUTED
0.7 Hz → 40 Hz: PRECOMPUTED
0.8 Hz → 40 Hz: PRECOMPUTED

0.3 Hz baseline filter: PRECOMPUTED

SINGLE-ECG FILTER TEST
----------------------------------------------------------------------
ECG ID:                    9
Input shape:               (5000, 12)
Fixed 0.5 Hz filter:       PASS
Adaptive filter:           PASS
Finite outputs:            PASS

Adaptive cutoffs for test ECG:
    I: 0.5 Hz
   II: 0.6 Hz
  III: 0.6 Hz
  AVR: 0.6 Hz
  AVL: 0.5 Hz
  AVF: 0.6 Hz
   V1: 0.5 Hz
   V2: 0.5 Hz
   V3: 0.6 Hz
   V4: 0.5 Hz
   V5: 0.6 Hz
   V6: 0.7 Hz

STEP 11H-2 STATUS: PASS
Filter coefficients precomputed.
Locked adaptive selection verified.
No ResNet evaluation performed.
No test-set predictions generated.
